# Pairwise TF-IDF Final Validation Analysis

This notebook reproduces the final Pairwise TF-IDF validation experiment from `notebooks/14_pairwise_tfidf_difference.ipynb` and generates validation-set artifacts for the final report and 10-minute video.

**Important:** Kaggle hidden-test labels are unavailable. All confusion matrices and error analyses in this notebook are conducted on the fixed local **Validation Set**, not on the Kaggle hidden test set.


In [ ]:
from pathlib import Path
import gc
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid')

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data' / 'processed'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
LOG_DIR = ROOT / 'outputs' / 'logs'
OOF_DIR = ROOT / 'outputs' / 'oof'
REPORT_DIR = ROOT / 'report'

for directory in [FIGURE_DIR, LOG_DIR, OOF_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('Project root:', ROOT)


## 1. Data loading and fixed validation split

This section intentionally reuses the data path, required columns, text cleaning, label validation, and split logic from notebook 14.


In [ ]:
train_path = DATA_DIR / 'train_eda.csv'

if not train_path.exists():
    raise FileNotFoundError(f'Required data file not found: {train_path}')

train = pd.read_csv(train_path, encoding='utf-8-sig')

required_train_columns = {
    'id', 'prompt_clean', 'response_a_clean', 'response_b_clean', 'label', 'label_name'
}
missing_train = sorted(required_train_columns - set(train.columns))
if missing_train:
    raise ValueError(f'train_eda.csv missing required columns: {missing_train}')

text_columns = ['prompt_clean', 'response_a_clean', 'response_b_clean']
for column in text_columns:
    train[column] = train[column].fillna('').astype(str)

train['label'] = pd.to_numeric(train['label'], errors='raise').astype(int)
if not train['label'].isin([0, 1, 2]).all():
    raise ValueError('Training labels must contain only 0, 1, and 2.')
if train['id'].duplicated().any():
    raise ValueError('Duplicate ids found in train_eda.csv.')

print('train shape:', train.shape)
print('train label distribution:')
print(train['label_name'].value_counts().sort_index())


In [ ]:
train_split, valid_split = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train['label'],
)
train_split = train_split.copy().reset_index(drop=True)
valid_split = valid_split.copy().reset_index(drop=True)

print('train_split shape:', train_split.shape)
print('valid_split shape:', valid_split.shape)

assert len(valid_split) == 11496
assert set(train_split['id']).isdisjoint(set(valid_split['id']))

print('train label distribution:')
print(train_split['label'].value_counts().sort_index())
print('validation label distribution:')
print(valid_split['label'].value_counts().sort_index())
print('train label_name distribution:')
print(train_split['label_name'].value_counts().sort_index())
print('validation label_name distribution:')
print(valid_split['label_name'].value_counts().sort_index())


## 2. Pairwise model reproduction

The feature construction, numeric features, A/B swap augmentation, TF-IDF parameters, and Logistic Regression parameters below are copied from the final Pairwise model implementation in notebook 14.


In [ ]:
LABEL_NAME_MAP = {0: 'A_win', 1: 'B_win', 2: 'tie'}
CLASS_NAMES = ['A_win', 'B_win', 'tie']
PROBABILITY_COLUMNS = ['prob_A_win', 'prob_B_win', 'prob_tie']

LEAKAGE_COLUMNS = {
    'model_a', 'model_b', 'winner_model_a', 'winner_model_b',
    'winner_tie', 'label', 'label_name'
}
print('Columns explicitly excluded from model features:', sorted(LEAKAGE_COLUMNS))

def swap_ab_dataframe(df):
    swapped = df.copy()
    swapped[['response_a_clean', 'response_b_clean']] = swapped[
        ['response_b_clean', 'response_a_clean']
    ].to_numpy()
    if 'label' in swapped.columns:
        swapped['label'] = swapped['label'].map({0: 1, 1: 0, 2: 2}).astype(int)
        if 'label_name' in swapped.columns:
            swapped['label_name'] = swapped['label'].map(LABEL_NAME_MAP)
    return swapped

swapped_train = swap_ab_dataframe(train_split)
train_aug = pd.concat([train_split, swapped_train], ignore_index=True)

print('Training rows before augmentation:', len(train_split))
print('Training rows after augmentation:', len(train_aug))
print('Augmented label distribution:')
print(train_aug['label'].value_counts().sort_index())


In [ ]:
prompt_vectorizer = TfidfVectorizer(
    analyzer='word',
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
    dtype=np.float32,
)
response_vectorizer = TfidfVectorizer(
    analyzer='word',
    max_features=60000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
    dtype=np.float32,
)

print('Fitting prompt vectorizer on original train_split only...')
prompt_vectorizer.fit(train_split['prompt_clean'])
response_fit_text = pd.concat(
    [train_split['response_a_clean'], train_split['response_b_clean']],
    ignore_index=True,
)
print('Fitting shared response vectorizer on original train_split responses only...')
response_vectorizer.fit(response_fit_text)
del response_fit_text
gc.collect()

print('Prompt vocabulary size:', len(prompt_vectorizer.vocabulary_))
print('Response vocabulary size:', len(response_vectorizer.vocabulary_))


In [ ]:
REFUSAL_TERMS = [
    'cannot', "can't", 'unable', "won't", 'sorry', 'not able',
    'cannot assist', 'cannot provide',
]
NUMBERED_LIST_PATTERN = re.compile(r'(?m)^\s*\d+\.\s+')

def sparse_memory_mb(matrix):
    if not hasattr(matrix, 'data'):
        return matrix.nbytes / (1024 ** 2)
    total_bytes = matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
    return total_bytes / (1024 ** 2)

def build_pairwise_text_features(
    df, fitted_prompt_vectorizer, fitted_response_vectorizer, dataset_name='dataset'
):
    x_prompt = fitted_prompt_vectorizer.transform(df['prompt_clean'])
    x_a = fitted_response_vectorizer.transform(df['response_a_clean'])
    x_b = fitted_response_vectorizer.transform(df['response_b_clean'])
    x_diff = x_a - x_b
    x_abs_diff = x_diff.copy()
    x_abs_diff.data = np.abs(x_abs_diff.data)
    x_sum = (x_a + x_b).multiply(np.float32(0.5))
    for feature_name, matrix in [
        ('X_prompt', x_prompt), ('X_a', x_a), ('X_b', x_b),
        ('X_diff', x_diff), ('X_abs_diff', x_abs_diff),
        ('X_sum', x_sum),
    ]:
        print(
            f'{dataset_name} {feature_name} shape: {matrix.shape}; '
            f'estimated memory: {sparse_memory_mb(matrix):.2f} MB'
        )
    del x_a, x_b
    gc.collect()
    x_text = hstack(
        [x_prompt, x_diff, x_abs_diff, x_sum],
        format='csr',
        dtype=np.float32,
    )
    print(
        f'{dataset_name} X_text shape: {x_text.shape}; '
        f'estimated memory: {sparse_memory_mb(x_text):.2f} MB'
    )
    del x_prompt, x_diff, x_abs_diff, x_sum
    gc.collect()
    return x_text

def count_list_markers(text):
    return text.count('\n- ') + text.count('\n* ') + len(NUMBERED_LIST_PATTERN.findall(text))

def count_refusal_terms(text):
    lowered = text.lower()
    return sum(lowered.count(term) for term in REFUSAL_TERMS)

def signed_log1p(values):
    values = np.asarray(values, dtype=np.float32)
    return np.sign(values) * np.log1p(np.abs(values))

def build_numeric_features(df):
    response_a = df['response_a_clean'].fillna('').astype(str)
    response_b = df['response_b_clean'].fillna('').astype(str)

    char_diff = response_a.str.len().to_numpy() - response_b.str.len().to_numpy()
    word_diff = (
        response_a.str.split().str.len().to_numpy()
        - response_b.str.split().str.len().to_numpy()
    )
    newline_diff = response_a.str.count('\n').to_numpy() - response_b.str.count('\n').to_numpy()
    list_diff = response_a.map(count_list_markers).to_numpy() - response_b.map(count_list_markers).to_numpy()
    code_diff = response_a.str.count('```').to_numpy() - response_b.str.count('```').to_numpy()
    refusal_diff = response_a.map(count_refusal_terms).to_numpy() - response_b.map(count_refusal_terms).to_numpy()
    question_diff = response_a.str.count(r'\?').to_numpy() - response_b.str.count(r'\?').to_numpy()
    exclamation_diff = response_a.str.count('!').to_numpy() - response_b.str.count('!').to_numpy()

    numeric = np.column_stack([
        signed_log1p(char_diff),
        np.log1p(np.abs(char_diff)),
        signed_log1p(word_diff),
        np.log1p(np.abs(word_diff)),
        signed_log1p(newline_diff),
        signed_log1p(list_diff),
        signed_log1p(code_diff),
        signed_log1p(refusal_diff),
        signed_log1p(question_diff),
        signed_log1p(exclamation_diff),
    ]).astype(np.float32)
    return numeric

NUMERIC_FEATURE_NAMES = [
    'char_len_diff_signed_log1p', 'char_len_abs_diff_log1p',
    'word_count_diff_signed_log1p', 'word_count_abs_diff_log1p',
    'newline_diff_signed_log1p', 'list_marker_diff_signed_log1p',
    'code_fence_diff_signed_log1p', 'refusal_diff_signed_log1p',
    'question_mark_diff_signed_log1p', 'exclamation_diff_signed_log1p',
]
print('Numeric feature count:', len(NUMERIC_FEATURE_NAMES))
print(NUMERIC_FEATURE_NAMES)


In [ ]:
print('Building pairwise text matrices...')
x_train_text = build_pairwise_text_features(
    train_aug, prompt_vectorizer, response_vectorizer, dataset_name='train_aug'
)
x_valid_text = build_pairwise_text_features(
    valid_split, prompt_vectorizer, response_vectorizer, dataset_name='valid_split'
)

print('Building and scaling numeric matrices...')
train_numeric_raw = build_numeric_features(train_aug)
valid_numeric_raw = build_numeric_features(valid_split)
numeric_scaler = StandardScaler()
train_numeric_scaled = numeric_scaler.fit_transform(train_numeric_raw).astype(np.float32)
valid_numeric_scaled = numeric_scaler.transform(valid_numeric_raw).astype(np.float32)
x_train_numeric = csr_matrix(train_numeric_scaled, dtype=np.float32)
x_valid_numeric = csr_matrix(valid_numeric_scaled, dtype=np.float32)

x_train_final = hstack([x_train_text, x_train_numeric], format='csr', dtype=np.float32)
x_valid_final = hstack([x_valid_text, x_valid_numeric], format='csr', dtype=np.float32)
y_train = train_aug['label'].to_numpy(dtype=np.int64)
y_valid = valid_split['label'].to_numpy(dtype=np.int64)

for name, matrix in [
    ('x_train_text', x_train_text), ('x_valid_text', x_valid_text),
    ('x_train_numeric', x_train_numeric), ('x_valid_numeric', x_valid_numeric),
    ('x_train_final', x_train_final), ('x_valid_final', x_valid_final),
]:
    print(f'{name} shape: {matrix.shape}; estimated memory: {sparse_memory_mb(matrix):.2f} MB')

del train_numeric_raw, valid_numeric_raw, train_numeric_scaled, valid_numeric_scaled
del x_train_text, x_valid_text, x_train_numeric, x_valid_numeric
gc.collect()


In [ ]:
best_pairwise_C = 0.1
print(f'Training final pairwise LogisticRegression with C={best_pairwise_C}...')
best_pairwise_model = LogisticRegression(
    C=best_pairwise_C,
    solver='saga',
    max_iter=800,
    tol=1e-3,
    n_jobs=-1,
    random_state=42,
)
best_pairwise_model.fit(x_train_final, y_train)
probs_valid = best_pairwise_model.predict_proba(x_valid_final)
pred_label = np.argmax(probs_valid, axis=1)


## 3. Validation probability checks and metrics


In [ ]:
assert probs_valid.shape == (11496, 3)
assert not np.isnan(probs_valid).any()
assert not np.isinf(probs_valid).any()
assert (probs_valid >= 0).all()
assert np.allclose(probs_valid.sum(axis=1), 1.0, atol=1e-5)

print('Class order: 0 = A_win, 1 = B_win, 2 = tie')
valid_log_loss = log_loss(y_valid, probs_valid, labels=[0, 1, 2])
valid_accuracy = accuracy_score(y_valid, pred_label)
valid_macro_f1 = f1_score(y_valid, pred_label, average='macro')

print('Actual validation metrics:')
print(f'- log loss: {valid_log_loss:.6f}')
print(f'- accuracy: {valid_accuracy:.6f}')
print(f'- macro F1: {valid_macro_f1:.6f}')

expected_metrics = {
    'log_loss': 1.017894,
    'accuracy': 0.488083,
    'macro_f1': 0.482842,
}
if abs(valid_log_loss - expected_metrics['log_loss']) > 0.001:
    raise RuntimeError(
        'WARNING: validation log loss differs from the expected notebook-14 result by more than 0.001. '
        f"Expected about {expected_metrics['log_loss']:.6f}, got {valid_log_loss:.6f}. "
        'Stopping before downstream analysis to avoid reporting incorrect artifacts.'
    )


## 4. Save per-sample validation predictions


In [ ]:
pairwise_valid_predictions = valid_split[[
    'id', 'label', 'label_name', 'prompt_clean', 'response_a_clean', 'response_b_clean'
]].copy()
pairwise_valid_predictions['pred_label'] = pred_label
pairwise_valid_predictions['pred_label_name'] = pairwise_valid_predictions['pred_label'].map(LABEL_NAME_MAP)
pairwise_valid_predictions['pred_confidence'] = probs_valid.max(axis=1)
pairwise_valid_predictions['true_label_prob'] = probs_valid[np.arange(len(y_valid)), y_valid]
pairwise_valid_predictions['prob_A_win'] = probs_valid[:, 0]
pairwise_valid_predictions['prob_B_win'] = probs_valid[:, 1]
pairwise_valid_predictions['prob_tie'] = probs_valid[:, 2]

prediction_columns = [
    'id', 'label', 'label_name', 'pred_label', 'pred_label_name',
    'pred_confidence', 'true_label_prob',
    'prob_A_win', 'prob_B_win', 'prob_tie',
]
pairwise_predictions_for_save = pairwise_valid_predictions[prediction_columns].copy()

assert len(pairwise_predictions_for_save) == 11496
assert not pairwise_predictions_for_save['id'].duplicated().any()
assert not pairwise_predictions_for_save.isna().any().any()

pairwise_oof_path = OOF_DIR / 'pairwise_tfidf_valid_predictions.csv'
pairwise_predictions_for_save.to_csv(pairwise_oof_path, index=False, encoding='utf-8-sig')
print('Saved:', pairwise_oof_path)


## 5. Confusion matrices (Validation Set)


In [ ]:
cm = confusion_matrix(y_valid, pred_label, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)

plt.figure(figsize=(7, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Pairwise TF-IDF Final Model Confusion Matrix\n(Validation Set)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
cm_path = FIGURE_DIR / 'pairwise_tfidf_confusion_matrix.png'
plt.savefig(cm_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.close()
with Image.open(cm_path) as img:
    img.verify()
print('Saved:', cm_path)
print('Confusion matrix:')
print(cm_df)


In [ ]:
cm_normalized = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True)
cm_normalized_df = pd.DataFrame(cm_normalized, index=CLASS_NAMES, columns=CLASS_NAMES)

plt.figure(figsize=(7, 6))
sns.heatmap(cm_normalized_df, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1, cbar=True)
plt.title('Normalized Pairwise TF-IDF Confusion Matrix\n(Validation Set)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
cm_norm_path = FIGURE_DIR / 'pairwise_tfidf_confusion_matrix_normalized.png'
plt.savefig(cm_norm_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.close()
with Image.open(cm_norm_path) as img:
    img.verify()
print('Saved:', cm_norm_path)
print(cm_normalized_df.round(3))


## 6. Classification metrics and per-class accuracy


In [ ]:
report_dict = classification_report(
    y_valid,
    pred_label,
    target_names=CLASS_NAMES,
    output_dict=True,
)
classification_report_df = pd.DataFrame(report_dict).T
classification_report_path = LOG_DIR / 'pairwise_classification_report.csv'
classification_report_df.to_csv(classification_report_path, encoding='utf-8-sig')
print('Saved:', classification_report_path)
display(classification_report_df)

per_class_recall = classification_report_df.loc[CLASS_NAMES, 'recall']
print('Per-class recall / class accuracy:')
for class_name, recall_value in per_class_recall.items():
    print(f'- {class_name} recall / class accuracy: {recall_value:.6f}')
print('In a single-label three-class task, each class recall is the share of examples from that true class correctly identified.')


## 7. True vs predicted label distribution


In [ ]:
true_distribution = pd.Series(y_valid).map(LABEL_NAME_MAP).value_counts().reindex(CLASS_NAMES, fill_value=0)
pred_distribution = pd.Series(pred_label).map(LABEL_NAME_MAP).value_counts().reindex(CLASS_NAMES, fill_value=0)
distribution_df = pd.DataFrame({
    'label_name': CLASS_NAMES,
    'true_count': true_distribution.values,
    'pred_count': pred_distribution.values,
})
distribution_df['true_ratio'] = distribution_df['true_count'] / distribution_df['true_count'].sum()
distribution_df['pred_ratio'] = distribution_df['pred_count'] / distribution_df['pred_count'].sum()

distribution_path = LOG_DIR / 'pairwise_prediction_distribution.csv'
distribution_df.to_csv(distribution_path, index=False, encoding='utf-8-sig')
print('Saved:', distribution_path)
display(distribution_df)

plot_df = distribution_df.melt(
    id_vars='label_name', value_vars=['true_count', 'pred_count'],
    var_name='distribution', value_name='count'
)
plt.figure(figsize=(8, 5))
sns.barplot(data=plot_df, x='label_name', y='count', hue='distribution')
plt.title('True vs Predicted Label Distribution\n(Pairwise TF-IDF Validation Set)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.tight_layout()
dist_fig_path = FIGURE_DIR / 'pairwise_true_vs_pred_distribution.png'
plt.savefig(dist_fig_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.close()
with Image.open(dist_fig_path) as img:
    img.verify()
print('Saved:', dist_fig_path)


## 8. High-confidence validation errors


In [ ]:
errors = pairwise_valid_predictions.loc[
    pairwise_valid_predictions['pred_label'] != pairwise_valid_predictions['label']
].copy()
errors = errors.sort_values('pred_confidence', ascending=False).reset_index(drop=True)

high_confidence_error_columns = [
    'id', 'label_name', 'pred_label_name', 'pred_confidence', 'true_label_prob',
    'prompt_clean', 'response_a_clean', 'response_b_clean',
]
high_confidence_errors_path = OOF_DIR / 'pairwise_high_confidence_errors.csv'
errors[high_confidence_error_columns].head(100).to_csv(
    high_confidence_errors_path, index=False, encoding='utf-8-sig'
)
print('Saved:', high_confidence_errors_path)
print('Validation error count:', len(errors))

def shorten_text(text, max_chars=300):
    text = str(text).replace('\r', ' ').replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + '...'

representative_pairs = [
    ('A_win', 'B_win'),
    ('B_win', 'A_win'),
    ('tie', 'A_win'),
    ('tie', 'B_win'),
    ('A_win', 'tie'),
    ('B_win', 'tie'),
]
typical_parts = []
for true_name, pred_name in representative_pairs:
    subset = errors[
        (errors['label_name'] == true_name) & (errors['pred_label_name'] == pred_name)
    ].head(3)
    if not subset.empty:
        typical_parts.append(subset)

typical_errors = pd.concat(typical_parts, ignore_index=True) if typical_parts else errors.head(15).copy()
typical_errors = typical_errors.head(18).copy()
typical_errors['prompt_short'] = typical_errors['prompt_clean'].map(shorten_text)
typical_errors['response_a_short'] = typical_errors['response_a_clean'].map(shorten_text)
typical_errors['response_b_short'] = typical_errors['response_b_clean'].map(shorten_text)

typical_error_columns = high_confidence_error_columns + [
    'prompt_short', 'response_a_short', 'response_b_short'
]
typical_errors_path = OOF_DIR / 'pairwise_typical_errors_for_report.csv'
typical_errors[typical_error_columns].to_csv(
    typical_errors_path, index=False, encoding='utf-8-sig'
)
print('Saved:', typical_errors_path)
display(typical_errors[['id', 'label_name', 'pred_label_name', 'pred_confidence', 'true_label_prob']])


## 9. Generate Markdown analysis report


In [ ]:
confusion_long = cm_df.stack().reset_index()
confusion_long.columns = ['true_label', 'pred_label', 'count']
confusion_long = confusion_long[confusion_long['true_label'] != confusion_long['pred_label']]
confusion_long = confusion_long.sort_values('count', ascending=False).reset_index(drop=True)

common_confusions_md = '\n'.join(
    f"- True {row.true_label} predicted {row.pred_label}: {int(row['count'])} samples"
    for _, row in confusion_long.head(6).iterrows()
)

per_class_md = '\n'.join(
    f"- {class_name}: precision={classification_report_df.loc[class_name, 'precision']:.6f}, "
    f"recall={classification_report_df.loc[class_name, 'recall']:.6f}, "
    f"F1={classification_report_df.loc[class_name, 'f1-score']:.6f}, "
    f"support={int(classification_report_df.loc[class_name, 'support'])}"
    for class_name in CLASS_NAMES
)

distribution_md = '\n'.join(
    f"- {row.label_name}: true={int(row.true_count)} ({row.true_ratio:.3%}), "
    f"predicted={int(row.pred_count)} ({row.pred_ratio:.3%})"
    for row in distribution_df.itertuples(index=False)
)

high_conf_error_md = '\n'.join(
    f"- id={row.id}: true={row.label_name}, predicted={row.pred_label_name}, "
    f"confidence={row.pred_confidence:.6f}, true_label_prob={row.true_label_prob:.6f}"
    for row in errors.head(10).itertuples(index=False)
)

report_text = f"""# Pairwise TF-IDF Final Validation Error Analysis

## 1. Final Pairwise validation metrics

This report analyzes the fixed local **Validation Set** produced with `train_test_split(test_size=0.2, random_state=42, stratify=train['label'])`.

- Best Pairwise C: {best_pairwise_C}
- Validation log loss: {valid_log_loss:.6f}
- Validation accuracy: {valid_accuracy:.6f}
- Validation macro F1: {valid_macro_f1:.6f}

## 2. Confusion matrix summary

Rows are true labels and columns are predicted labels.

{cm_df.to_markdown()}

## 3. Per-class precision / recall / F1

{per_class_md}

In this single-label three-class setup, per-class recall is also the class-level accuracy for examples whose true label is that class.

## 4. Most common confusion directions

{common_confusions_md}

## 5. Predicted label distribution

{distribution_md}

## 6. High-confidence error analysis

The highest-confidence incorrect validation predictions are:

{high_conf_error_md}

Full high-confidence error samples are saved to `outputs/oof/pairwise_high_confidence_errors.csv`, and a shorter human-readable subset is saved to `outputs/oof/pairwise_typical_errors_for_report.csv`.

## 7. Limitations

- This is local validation performance, not a labeled Kaggle hidden-test analysis.
- Kaggle hidden test labels are unavailable, so confusion-matrix and error analyses are conducted on the fixed local validation split.
- The Pairwise TF-IDF model depends on lexical overlap and engineered response-difference features; it may miss semantic quality differences that are not well represented by TF-IDF.
- SAGA optimization can have very small numerical variation across environments, so metric checks allow only a narrow tolerance before stopping downstream report generation.

## Local validation vs Kaggle hidden-test Public Score

The confusion matrices, classification report, and error samples above are based only on the local **Validation Set**. Kaggle hidden-test labels are unavailable, so the hidden-test leaderboard can provide log loss but not a confusion matrix or per-sample error analysis.

- Kaggle Public Score = 1.02159
- Leaderboard rank at screenshot time = 67
"""

report_path = REPORT_DIR / 'pairwise_final_error_analysis.md'
report_path.write_text(report_text, encoding='utf-8')
print('Saved:', report_path)


## 10. Final summary printout


In [ ]:
print('Actual validation metrics:')
print(f'- log loss: {valid_log_loss:.6f}')
print(f'- accuracy: {valid_accuracy:.6f}')
print(f'- macro F1: {valid_macro_f1:.6f}')
print('\nConfusion matrix')
print(cm_df)
print('\nPer-class recall')
for class_name, recall_value in per_class_recall.items():
    print(f'- {class_name}: {recall_value:.6f}')
print('\nSaved files:')
for path in [
    pairwise_oof_path,
    cm_path,
    cm_norm_path,
    classification_report_path,
    distribution_path,
    dist_fig_path,
    high_confidence_errors_path,
    typical_errors_path,
    report_path,
]:
    print(f'- {path.relative_to(ROOT)}')

print('\nPairwise final validation analysis finished successfully.')

del x_train_final, x_valid_final
gc.collect()
